In [1]:
import pandas as pd
import numpy as np 
import os
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier

In [2]:
INPUT_PATH = "https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz"
OUTPUT_PATH = '../data/openfoodfacts_regression_clean.csv'
 
# Colonnes pour les features et la cible numérique
nutriscore_cols = [
    'energy_100g', 'fat_100g', 'saturated-fat_100g',
    'sugars_100g', 'fiber_100g', 'proteins_100g', 'carbohydrates_100g',
    'salt_100g'
]
 
# Changement : on vise le SCORE (numérique)
target = 'nutriscore_score'
identity_cols = ['code', 'product_name', 'nutriscore_grade'] # On garde le grade juste pour info
 
cols = nutriscore_cols + [target] + identity_cols
def process_data(file_path, cols, chunk_size=20000):
    reader = pd.read_csv(
        file_path, compression='gzip', sep='\t',
        on_bad_lines='skip', chunksize=chunk_size,
        low_memory=False, usecols=cols
    )
 
    clean_chunks = []
    print("Lecture des blocs...")
    
    for i, chunk in enumerate(reader):
 
        temp_chunk = chunk.copy()
 
        # 1. Nettoyage de la Cible (Target)
        # On supprime les lignes où le score numérique manque
        temp_chunk = temp_chunk.dropna(subset=[target])
        cols_to_convert = nutriscore_cols + [target]
 
        # 2. Conversion numérique des nutriments
        for col in cols_to_convert:
            temp_chunk[col] = pd.to_numeric(temp_chunk[col], errors='coerce')
 
        # Filtre de réalisme pour le score (Officiel : -15 à +40)
        temp_chunk = temp_chunk[(temp_chunk[target] >= -15) & (temp_chunk[target] <= 40)]            
 
        # Remplissage des nutriments manquants par 0 (Imputation)
        temp_chunk[nutriscore_cols] = temp_chunk[nutriscore_cols].fillna(0)
        
        # 3. Filtres Outliers (0-100g et Energie)
        for col in nutriscore_cols:
            if col != 'energy_100g':
                temp_chunk = temp_chunk[(temp_chunk[col] >= 0) & (temp_chunk[col] <= 100)]
        
        temp_chunk = temp_chunk[(temp_chunk['energy_100g'] >= 0) & (temp_chunk['energy_100g'] < 4000)]
 
        # 4. Gestion de l'identité
        temp_chunk['product_name'] = temp_chunk['product_name'].fillna('Unknown Product')
        temp_chunk = temp_chunk.drop_duplicates(subset=['code'])
 
         # 5. Stockage du bloc propre
        if not temp_chunk.empty:
            clean_chunks.append(temp_chunk)
        
        if i % 10 == 0:
            print(f"Bloc {i} chargé...")
    
    print("Fusion de tous les blocs...")
    df_final = pd.concat(clean_chunks, ignore_index=True)
    
    return df_final
 
 
# Lancement
df_raw = process_data(INPUT_PATH, cols)

Lecture des blocs...
Bloc 0 chargé...
Bloc 10 chargé...
Bloc 20 chargé...
Bloc 30 chargé...
Bloc 40 chargé...
Bloc 50 chargé...
Bloc 60 chargé...
Bloc 70 chargé...
Bloc 80 chargé...
Bloc 90 chargé...
Bloc 100 chargé...
Bloc 110 chargé...
Bloc 120 chargé...
Bloc 130 chargé...
Bloc 140 chargé...
Bloc 150 chargé...
Bloc 160 chargé...
Bloc 170 chargé...
Bloc 180 chargé...
Bloc 190 chargé...
Bloc 200 chargé...
Bloc 210 chargé...
Bloc 220 chargé...
Fusion de tous les blocs...


In [3]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df_raw.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')

print(f"Fichier sauvegardé ici : {OUTPUT_PATH}")

Fichier sauvegardé ici : ../data/openfoodfacts_regression_clean.csv


In [5]:
df_raw

,code,product_name,nutriscore_score,nutriscore_grade,energy_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g
0,7,granola Bio le Chocolaté,4.0,c,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00
1,8,Unknown Product,6.0,c,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00
2,9,xytitol pastilles,-11.0,a,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00
3,13,Powdered peanut butter,3.0,c,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00
4,15,Madeleines ChocoLait,20.0,e,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...
1351778,9999999175305,Erdbeerkuchen 1019g tiefgefroren,13.0,d,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00
1351779,99999995,Steak haché 5%,1.0,b,550.5,5.0,2.3,0.0,0.0,0.0,21.5,0.18
1351780,9999999916298,Beurre de cacahuète bio,-9.0,a,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00
1351781,9999999999970,Unknown Product,4.0,c,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00
